## viz 

In [ ]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# ============================================================
# Inputs
# ============================================================

root = Path(r"C:\Users\yingjiel\Documents\mongolia-mining")
dir = root / "data"

mining_fp = root / 'data' / 'es_footprint_outputs' / "es_footprint_Maus2022.gpkg"
mongolia_boundary_fp = dir / "ne_50m_admin_0_countries_MNG.shp"
out_dir = Path(r"D:\path\to\ppt_figures")
out_dir.mkdir(parents=True, exist_ok=True)

# List your services here:
# key = short service name used in output filename
# value = (mean column, sum column)
SERVICES = {
    "water": ("water_mean", "water_sum"),
    "carbon": ("carbon_mean", "carbon_sum"),
    "habitat": ("habitat_mean", "habitat_sum"),
}

# Optional Mongolia equal-area CRS for area-safe geometry ops
# plotting will later be converted to 4326
EQUAL_AREA_CRS = 8857   # Equal Earth
PLOT_CRS = 4326

# ============================================================
# Load data
# ============================================================
mining = gpd.read_file(mining_fp)

boundary = None
if mongolia_boundary_fp.exists():
    boundary = gpd.read_file(mongolia_boundary_fp)

# Reproject to equal-area CRS first
if mining.crs is None:
    raise ValueError("Mining layer has no CRS defined.")

mining = mining.to_crs(EQUAL_AREA_CRS)

if boundary is not None:
    boundary = boundary.to_crs(EQUAL_AREA_CRS)

# ============================================================
# Convert polygons to representative points
# representative_point() stays inside polygon
# ============================================================
points = mining.copy()
points["geometry"] = points.geometry.representative_point()

# Reproject to plotting CRS
points = points.to_crs(PLOT_CRS)
if boundary is not None:
    boundary = boundary.to_crs(PLOT_CRS)

# ============================================================
# Helper to make marker sizes from sum column
# ============================================================
def make_plot_sizes(values, min_size=10, max_size=250):
    """
    Convert numeric values to marker sizes for matplotlib.
    Uses sqrt scaling to reduce dominance of very large values.
    """
    vals = np.asarray(values, dtype=float)

    # handle all-zero / constant / missing cases
    vals = np.where(np.isfinite(vals), vals, np.nan)
    if np.all(np.isnan(vals)):
        return np.full(len(vals), min_size)

    # set negative values to zero for marker sizing
    vals = np.where(vals < 0, 0, vals)

    scaled = np.sqrt(vals)

    vmin = np.nanmin(scaled)
    vmax = np.nanmax(scaled)

    if np.isclose(vmin, vmax):
        return np.full(len(vals), (min_size + max_size) / 2)

    sizes = min_size + (scaled - vmin) / (vmax - vmin) * (max_size - min_size)
    return sizes


# ============================================================
# Helper to add size legend
# ============================================================
def add_size_legend(ax, raw_values, size_values, title="Sum"):
    raw_values = np.asarray(raw_values, dtype=float)
    raw_values = raw_values[np.isfinite(raw_values)]
    raw_values = raw_values[raw_values >= 0]

    if len(raw_values) == 0:
        return

    # choose representative legend values
    qs = np.quantile(raw_values, [0.25, 0.5, 0.75, 0.95])
    legend_raw = np.unique(np.round(qs, 2))

    # convert legend raw values to plot sizes using same scaling
    legend_sizes = make_plot_sizes(legend_raw, min_size=size_values.min(), max_size=size_values.max())

    handles = [
        plt.scatter([], [], s=s, facecolor="lightgray", edgecolor="black", alpha=0.8)
        for s in legend_sizes
    ]
    labels = [f"{v:,.2f}" for v in legend_raw]

    ax.legend(
        handles,
        labels,
        title=title,
        loc="lower left",
        frameon=True
    )


# ============================================================
# Plot one map per service
# ============================================================
for service_name, (mean_col, sum_col) in SERVICES.items():

    if mean_col not in points.columns or sum_col not in points.columns:
        print(f"Skipping {service_name}: missing {mean_col} or {sum_col}")
        continue

    plot_gdf = points[[mean_col, sum_col, "geometry"]].copy()

    # Drop rows with both missing
    plot_gdf = plot_gdf[plot_gdf[mean_col].notna() | plot_gdf[sum_col].notna()].copy()
    if plot_gdf.empty:
        print(f"Skipping {service_name}: no valid data")
        continue

    plot_gdf["plot_size"] = make_plot_sizes(plot_gdf[sum_col].values, min_size=12, max_size=220)

    fig, ax = plt.subplots(figsize=(11, 8))

    # Base boundary
    if boundary is not None:
        boundary.boundary.plot(ax=ax, color="lightgray", linewidth=0.8)

    # Scatter-like point map using GeoPandas
    plot_gdf.plot(
        ax=ax,
        column=mean_col,
        markersize=plot_gdf["plot_size"],
        cmap="viridis",
        alpha=0.75,
        edgecolor="black",
        linewidth=0.2,
        legend=True,
        legend_kwds={"label": f"{service_name.capitalize()} mean"}
    )

    # Add size legend for sum
    add_size_legend(
        ax,
        raw_values=plot_gdf[sum_col].values,
        size_values=plot_gdf["plot_size"].values,
        title=f"{service_name.capitalize()} sum"
    )

    # Add total number
    total_n = len(plot_gdf)
    ax.text(
        0.02, 0.98,
        f"Sites: {total_n}",
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="none")
    )

    ax.set_title(f"Mining sites in Mongolia: {service_name.capitalize()}", fontsize=16)
    ax.set_axis_off()

    out_png = out_dir / f"mining_points_{service_name}_mean_sum.png"
    plt.savefig(out_png, dpi=500, bbox_inches="tight", facecolor="white")
    plt.close()

    print(f"Saved: {out_png}")